In [10]:
# =========================
# 1. Import Libraries
# =========================
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, classification_report


# =========================
# 2. Load Dataset
# =========================
df = pd.read_csv(r"C:\Users\ABHAY SINGH\OneDrive\Documents\Desktop\AI TWIN\ultimate_student_productivity_dataset_5000.csv")

print("Initial Data:")
print(df.head())


# =========================
# 3. Drop Unnecessary Columns
# =========================
if "student_id" in df.columns:
    df.drop("student_id", axis=1, inplace=True)


# =========================
# 4. Handle Categorical Variables
# =========================
df = pd.get_dummies(df, drop_first=True)


# =========================
# 5. Define Targets
# =========================
y_productivity = df["productivity_score"]
y_burnout = df["burnout_level"]

# Convert burnout into category
y_burnout_cat = pd.cut(
    y_burnout,
    bins=[0, 40, 70, 100],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)


# =========================
# 6. Define Features
# =========================
X = df.drop([
    "productivity_score",
    "burnout_level",
    "exam_score",
    "focus_index"
], axis=1)


# Save column structure
joblib.dump(X.columns, "columns1.pkl")


# =========================
# 7. Train-Test Split
# =========================
X_train, X_test, y_prod_train, y_prod_test, y_burn_train, y_burn_test = train_test_split(
    X,
    y_productivity,
    y_burnout_cat,
    test_size=0.2,
    random_state=42
)


# =========================
# 8. Feature Scaling
# =========================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "scaler1.pkl")


# =========================
# 9. Train Productivity Model
# =========================
reg_model1 = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reg_model1.fit(X_train_scaled, y_prod_train)

y_pred_prod = reg_model1.predict(X_test_scaled)

print("\n📈 Productivity Model Results")
print("R2 Score:", round(r2_score(y_prod_test, y_pred_prod), 3))
print("MAE:", round(mean_absolute_error(y_prod_test, y_pred_prod), 3))

joblib.dump(reg_model1, "productivity_model1.pkl")


# =========================
# 10. Train Burnout Model
# =========================
clf_model1 = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

clf_model1.fit(X_train_scaled, y_burn_train)

y_pred_burn = clf_model1.predict(X_test_scaled)

print("\n🔥 Burnout Model Results")
print("Accuracy:", accuracy_score(y_burn_test, y_pred_burn))
print(classification_report(y_burn_test, y_pred_burn))

joblib.dump(clf_model1, "burnout_model1.pkl")


# =========================
# 11. Test Prediction
# =========================

print("\n=========================")
print("🔮 Testing with New User")
print("=========================")


user_input = {
    "age": 21,
    "study_hours": 2,
    "self_study_hours": 0,
    "online_classes_hours": 0,
    "social_media_hours": 9,
    "gaming_hours": 3,
    "screen_time_hours": 15,
    "sleep_hours": 2,
    "exercise_minutes": 30,
    "caffeine_intake_mg": 0,
    "mental_health_score": 0,
    "gender": "Male",
    "academic_level": "UG",
    "part_time_job": "No",
    "internet_quality": "Poor",
    "upcoming_deadline": "Yes"
}


# Convert to dataframe
input_df = pd.DataFrame([user_input])


# Apply encoding
input_df = pd.get_dummies(input_df)


# Load training columns
columns = joblib.load("columns1.pkl")


# Match columns
input_df = input_df.reindex(columns=columns, fill_value=0)


# Scale input
input_scaled = scaler.transform(input_df)


# Predict
prod_pred = reg_model1.predict(input_scaled)
burn_pred = clf_model1.predict(input_scaled)


print("\n📈 Predicted Productivity:", round(prod_pred[0], 2))
print("🔥 Predicted Burnout Level:", burn_pred[0])

Initial Data:
   student_id  age gender academic_level  study_hours  self_study_hours  \
0           1   18  Other    High School         7.64              1.56   
1           2   18  Other    High School         2.21              2.22   
2           3   22   Male    High School         3.45              0.00   
3           4   17  Other    High School         5.75              2.08   
4           5   19  Other    High School         6.83              1.72   

   online_classes_hours  social_media_hours  gaming_hours  sleep_hours  ...  \
0                  2.20                3.05          2.19         6.52  ...   
1                  2.10                1.65          2.55         5.97  ...   
2                  0.29                1.34          2.08         8.39  ...   
3                  3.01                2.27          2.20         6.31  ...   
4                  3.33                2.65          0.70         8.01  ...   

   exercise_minutes  caffeine_intake_mg  part_time_job  upco